## REQUIRES ENV FILE WITH AGOL CREDENTIALS

In [ ]:
from typing import List, Sequence, Tuple, Literal
from math import hypot

from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform
import pyproj


from typing import List, Sequence, Tuple, Literal
from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    MultiLineString as ShapelyMultiLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon as ShapelyMultiPolygon,
)
from shapely.ops import transform, unary_union
import pyproj


from shapely.geometry import (
    Point as ShapelyPoint,
    LineString as ShapelyLineString,
    Polygon as ShapelyPolygon,
    MultiPolygon,
)
from shapely.ops import transform, unary_union, polygonize
import pyproj
import json
import requests
import math
import streamlit as st
import logging
from shapely.geometry import LineString
from shapely.ops import unary_union, linemerge
from typing import Tuple, Optional, List, Dict, Any

In [ ]:
import os
import requests
from dotenv import load_dotenv

# Load .env file once at app startup (safe to call multiple times)
load_dotenv()


def get_agol_token() -> str:
    """
    Generates an authentication token for ArcGIS Online using a username and password
    stored in environment variables.

    Environment variables required:
        - AGOL_USERNAME
        - AGOL_PASSWORD

    Returns:
        str: A valid authentication token used to make authorized API requests.

    Raises:
        ValueError: If credentials are missing or authentication fails.
        ConnectionError: If requests cannot reach the AGOL endpoint.
    """
    username = os.getenv("AGOL_USERNAME")
    password = os.getenv("AGOL_PASSWORD")

    if not username or not password:
        raise ValueError("Missing AGOL_USERNAME or AGOL_PASSWORD in environment variables.")

    url = "https://www.arcgis.com/sharing/rest/generateToken"

    data = {
        "username": username,
        "password": password,
        "referer": "https://www.arcgis.com",
        "f": "json"
    }

    try:
        response = requests.post(url, data=data)

        if response.status_code != 200:
            raise Exception(
                f"Request failed with status code {response.status_code}: {response.text}"
            )

        token_data = response.json()

        if "token" in token_data:
            return token_data["token"]
        elif "error" in token_data:
            raise ValueError(
                f"Authentication failed: {token_data['error']['message']}"
            )
        else:
            raise ValueError("Unexpected response format: Token not found.")

    except requests.exceptions.RequestException as e:
        raise ConnectionError(f"Failed to connect to ArcGIS Online: {e}")


In [ ]:
def query_record(url: str, layer: int, where: str, fields="*", return_geometry=False):
    """
    Executes an SQL-style query against an ArcGIS REST API layer and returns matching records.

    Parameters:
        url: str
            FeatureServer base URL (may or may not already include a layer segment).
        layer: int
            Layer index when url is a FeatureServer root.
        where: str
            SQL-like filter clause (e.g., "GlobalID='...'" or "1=1").
        fields: str
            outFields string. "*" requests all fields.
        return_geometry: bool
            Whether to return geometry in results.

    Returns:
        list: List of 'features' from the ArcGIS REST response.
    """
    token = get_agol_token()
    if not token:
        raise ValueError("Authentication failed: Invalid token.")


    # If the URL already ends with the layer number, don't add it again
    if url.split("/")[-1].isdigit():
        query_url = f"{url}/query"
    else:
        query_url = f"{url}/{layer}/query"

    params = {
        "where": where,
        "outFields": fields,
        "returnGeometry": str(return_geometry).lower(),
        "outSR": 4326,
        "f": "json",
        "token": token
    }

    response = requests.get(query_url, params=params)
    if response.status_code != 200:
        raise Exception(
            f"Request failed with status code {response.status_code}: {response.text}"
        )

    data = response.json()
    if "error" in data:
        raise Exception(
            f"API Error: {data['error']['message']} - {data['error'].get('details', [])}"
        )

    return data.get("features", [])

In [ ]:
token = get_agol_token()

traffic_impact_form_url = "https://services.arcgis.com/r4A0V7UzH9fcLVvv/arcgis/rest/services/service_885f75157e3042f2bf3c1cfec1a8094e/FeatureServer"

traffic_impact_form_layers = {
'traffic_impacts_form_layer': 0
}




In [22]:
events = query_record(
    traffic_impact_form_url,
    traffic_impact_form_layers ['traffic_impacts_form_layer'],
    '1=1',
    return_geometry=True
)

apex_guid_list = []

for event in events:
    apex_guid =  event['attributes']['APEX_GUID']
    if apex_guid not in apex_guid_list:
        apex_guid_list.append(apex_guid)


for guid in apex_guid_list:
    project_submissions = query_record(
                            traffic_impact_form_url,
                            traffic_impact_form_layers ['traffic_impacts_form_layer'],
                            f"APEX_GUID='{guid}'",
                            return_geometry=True
                        )
    
    
    # records = your list of features
    latest_creation_date = max(
        feature["attributes"]["CreationDate"]
        for feature in project_submissions
    )

    not_latest_records = [
        feature['attributes']['objectid']
        for feature in project_submissions
        if feature["attributes"]["CreationDate"] != latest_creation_date
    ]

   
    if len(not_latest_records) != 0:

        updates = []
        
        for objectid in not_latest_records:
            package = {
                "attributes": {
                    "objectid": objectid,
                    'Latest_Entry': "No"
                    
                }
            }

            updates.append(package)

        print(updates)

        # --------------------------------------------------
        # APPLY EDITS (UPDATES)
        # --------------------------------------------------

        apply_edits_url = (
            f"{traffic_impact_form_url}/{traffic_impact_form_layers ['traffic_impacts_form_layer']}/applyEdits"
        )

        payload = {
            "f": "json",
            "updates": json.dumps(updates),
            "token": token
        }

        response = requests.post(apply_edits_url, data=payload)

        print("Status:", response.status_code)
        print(json.dumps(response.json(), indent=2))


[{'attributes': {'objectid': 123, 'Latest_Entry': 'No'}}]
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 123,
      "uniqueId": 123,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
[{'attributes': {'objectid': 125, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 235, 'Latest_Entry': 'No'}}]
Status: 200
{
  "addResults": [],
  "updateResults": [
    {
      "objectId": 125,
      "uniqueId": 125,
      "globalId": null,
      "success": true
    },
    {
      "objectId": 235,
      "uniqueId": 235,
      "globalId": null,
      "success": true
    }
  ],
  "deleteResults": []
}
[{'attributes': {'objectid': 126, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 200, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 233, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 1247, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 1261, 'Latest_Entry': 'No'}}, {'attributes': {'objectid': 1271, 'Latest_Entry': 'No'}}]


In [ ]:
not_latest_records